# Đề cập lại bài toán
- Bài toán ban đầu của nhóm là phân tích và dự đoán giá nhà ở 2 thành phố lớn là Hồ Chí Minh và Hà Nội nên nhóm sẽ xây dựng mô hình để dự đoán giá nhà


# Xây dựng mô hình

## Chuẩn bị dữ liệu chung cho cả 2 mô hình

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/unified/preprocessed_merged.csv')

df = df[(df['gia'] >= 0.5) & (df['gia'] <=500) & (df['so_tang'] <= 16) & (df['phong_ngu'] <= 30) & (df['phong_tam'] <= 30) & (df['dien_tich_dat'] <= 3000) & (df['dien_tich_dat'] > 5) & df['gia_tren_m2'] > 0]

Chia tập dữ liệu thành 80/20 với tương ứng với tập train/test 

In [8]:
# Giả sử cột target là 'price'
X = df.drop('gia', axis=1)
y = df['gia']

# Chia dữ liệu thành tập train và test với tỉ lệ 80-20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Mô hình 1
- Mô hình 1 sẽ dùng một biến kết hợp đó là quan x dien_tich_dat và cộng với diem_sinh_loi trong đó
- quan x dien_tich_dat là One hot encoding quận sau đó nhân với diện tích đất để ra biến mới, lý do kết hợp như này vì phần lớn giá nhà ở các trung tâm như Hà Nội và Hồ Chí Minh phụ thuộc rất nhiều vào vị trí, có những căn nhà vài mét vuông nhưng ở vị trí đắt đỏ lại có thể bán được vài tỷ [[1]](https://cafef.vn/con-duong-dat-do-bac-nhat-viet-nam-gia-nha-len-toi-3-ty-dong-m2-188240921095104835.chn) nên ý tưởng là thay vì chỉ quan tâm đến giá theo diện tích đất chung ta sẽ quan tâm đến giá theo diện tích đất theo từng quận sau đó cộng với diem_sinh_loi vì biến này được tính theo số phòng ngủ,số phòng tắm, số tầng,... vì thế cũng sẽ khái quát được độ lớn của ngôi nhà nên cũng sẽ ảnh hưởng đến giá.
- Do vậy 2 feature này sẽ tổng quát được vị trí theo giá và độ lớn của ngôi nhà



In [9]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# --- Tạo feature như mô tả ---

# One-hot encoding cho "quan"
quan_dummies = pd.get_dummies(X_train['quan'], prefix='quan')
quan_dummies_test = pd.get_dummies(X_test['quan'], prefix='quan')


# Đảm bảo các cột quận giống nhau giữa train và test (tránh missing columns)
quan_dummies, quan_dummies_test = quan_dummies.align(quan_dummies_test, join='left', axis=1, fill_value=0)

# feature: quan x dien_tich_dat
for col in quan_dummies.columns:
    X_train[f"{col}_x_dien_tich_dat"] = quan_dummies[col] * X_train['dien_tich_dat']
    X_test[f"{col}_x_dien_tich_dat"] = quan_dummies_test[col] * X_test['dien_tich_dat']


# Tạo feature đầu ra cuối cùng: tất cả các "quan x dien_tich_dat" + diem_sinh_loi + các feature phap_ly + các feature thanh_pho
feature_cols = [c for c in X_train.columns if '_x_dien_tich_dat' in c]
feature_cols.append('diem_sinh_loi')

X_train_features = X_train[feature_cols].copy()
X_test_features = X_test[feature_cols].copy()

# --- So sánh mô hình ---

results = {}
from sklearn.metrics import mean_absolute_error

# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_features, y_train)
y_pred_lr = lin_reg.predict(X_test_features)
results['Linear Regression'] = {
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_lr)),
    'MAE': mean_absolute_error(y_test, y_pred_lr),
    'R2': r2_score(y_test, y_pred_lr)
}

# Random Forest
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_features, y_train)
y_pred_rf = rf.predict(X_test_features)
results['Random Forest'] = {
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_rf)),
    'MAE': mean_absolute_error(y_test, y_pred_rf),
    'R2': r2_score(y_test, y_pred_rf)
}

# Hiển thị kết quả
print("So sánh 2 thuật toán:")
for model_name, metrics in results.items():
    print(f"{model_name}:")
    print(f"    RMSE: {metrics['RMSE']:.2f}")
    print(f"    MAE:  {metrics['MAE']:.2f}")
    print(f"    R2:   {metrics['R2']:.4f}")



So sánh 2 thuật toán:
Linear Regression:
    RMSE: 13.61
    MAE:  6.59
    R2:   0.6823
Random Forest:
    RMSE: 14.65
    MAE:  5.41
    R2:   0.6316


## Nhận xét chi tiết về kết quả so sánh 2 thuật toán của mô hình 1

### 1. Phân tích các chỉ số đánh giá của 2 thuật toán

#### 1.1. Root Mean Squared Error (RMSE)
**Linear Regression: 13.61 tỷ | Random Forest: 14.65 tỷ**

- Linear Regression có RMSE thấp hơn so với Random Forest, cho thấy mô hình tuyến tính dự đoán chính xác hơn về tổng thể.
- RMSE là chỉ số nhạy cảm với các giá trị outliers do sử dụng bình phương sai số. Việc Linear Regression có RMSE thấp hơn cho thấy mô hình này xử lý tốt các trường hợp có sai số lớn
- Với ngữ cảnh giá nhà, RMSE 13.61 tỷ có nghĩa là trung bình mô hình có thể sai lệch khoảng 13-14 tỷ đồng. Đây là mức chấp nhận được khi xét trong khoảng giá từ 0.5 - 500 tỷ

#### 1.2. Mean Absolute Error (MAE)
**Linear Regression: 6.59 tỷ | Random Forest: 5.41 tỷ**

- Random Forest có MAE thấp hơn so với Linear Regression - đây là ưu điểm đáng kể
- MAE phản ánh sai số trung bình thực tế mà người dùng sẽ gặp phải. Với Random Forest, trung bình một dự đoán sai lệch khoảng 5.41 tỷ, trong khi Linear Regression sai lệch 6.59 tỷ
- Sự chênh lệch này (1.18 tỷ) là có ý nghĩa trong thực tế, đặc biệt với các căn nhà giá trung bình
- MAE thấp hơn của Random Forest cho thấy mô hình này dự đoán tốt hơn với **phần lớn các trường hợp thông thường**, ít bị ảnh hưởng bởi outliers

#### 1.3. R² Score (Hệ số xác định)
**Linear Regression: 0.6823 | Random Forest: 0.6316**

- Linear Regression giải thích được **68.23%** biến động của giá nhà, cao hơn Random Forest (**63.16%**)
- Chênh lệch **5.07%** trong R² là đáng kể, cho thấy Linear Regression nắm bắt được mối quan hệ tổng thể tốt hơn
- R² > 0.6 cho cả hai mô hình là kết quả khá tốt trong bài toán dự đoán giá bất động sản, vốn chịu ảnh hưởng của nhiều yếu tố không quan sát được
- Việc Linear Regression đạt R² cao hơn cho thấy **feature engineering đã thực hiện (quan × diện_tích_đất) có mối quan hệ tuyến tính mạnh với giá nhà**

### 2. Kết luận về hiệu suất từng thuật toán

#### 2.1. Linear Regression

**Điểm mạnh:** Độ chính xác tổng thể cao, xử lý tốt các giá trị cực đoan, mô hình đơn giản dễ giải thích và triển khai

**Điểm yếu:** Sai số trung bình (MAE) cao hơn Random Forest với các trường hợp thông thường

#### 2.2. Random Forest

**Điểm mạnh:** Sai số trung bình thấp, dự đoán tốt với các trường hợp phổ biến

**Điểm yếu:** R² thấp hơn, RMSE cao hơn, không phù hợp với feature design hiện tại, khó giải thích

#### **Hiệu suất tổng thể: Linear Regression tốt hơn Random Forest**

### 3. Kết luận chung và lựa chọn thuật toán

**Mô hình được chọn: Linear Regression**

Dựa trên phân tích trên, **Linear Regression** là lựa chọn tối ưu vì:

1. **Hiệu suất tổng thể vượt trội**: R² cao hơn 5.07%, RMSE thấp hơn 1.04 - đây là những cải thiện đáng kể
2. **Xử lý tốt mọi trường hợp**: Không chỉ tốt với trường hợp thông thường mà còn xử lý tốt outliers và giá nhà cao
3. **Phù hợp với bản chất bài toán**: Feature engineering đã thiết kế dựa trên giả định tuyến tính về mối quan hệ giá - vị trí - diện tích, và kết quả chứng minh giả định này đúng
4. **Tính thực tiễn**: Mô hình đơn giản, dễ triển khai
5. **Độ tin cậy**: Ít rủi ro overfitting, generalize tốt với dữ liệu mới

Random Forest tuy có MAE thấp hơn nhưng không đủ để bù đắp cho những điểm yếu về R² và RMSE. Hơn nữa, tính đơn giản và khả năng giải thích của Linear Regression là lợi thế lớn trong thực tế khi cần thuyết phục stakeholders hoặc giải thích kết quả cho người dùng cuối.

**Kết quả cuối cùng**: Linear Regression với **R² = 0.6823, RMSE = 13.61 tỷ, MAE = 6.59 tỷ** sẵn sàng cho giai đoạn triển khai và sử dụng thực tế.